# KernelPack RAG — Retrieval Experiments (v1→v3)

Baseline hybrid retrieval pipeline on `kernelpack-python`. Each version
isolates one change, runs the same eval, and diagnoses failure modes before
moving to the next knob.

## Experiment Versions

| Version | Scope | Change | Hypothesis | recall@3 |
|---------|-------|--------|------------|----------|
| v1 | full index | baseline — raw source, MIN_LINES=5 | — | 1/10 (10%) |
| v2 | full index | MIN_LINES=1 | short functions are real targets | 2/10 (20%) |
| v3a | solvers only | no summaries (control) | smaller index = less noise | 2/10 (20%) |
| v3 | solvers only | LLM summaries prepended | summaries close the vocabulary gap | 3/10 (30%) |

## Key Findings

- 8/10 targets land in the top-10 neighborhood after summaries — the bottleneck is ranking, not vocabulary gap.
- MIN_LINES filtering drops real targets, but lowering it adds noise. Line count is the wrong filter.
- A cross-encoder reranker is the highest-leverage next step.

## Contents

| Section | Description |
|---------|-------------|
| Setup | Deps, chunking, model, helpers, eval data |
| v1 — Baseline | Establish the recall@3 floor |
| v2 — MIN_LINES=1 | Recover short-function targets |
| v3a — Solvers-only | Isolate scoping effect (control) |
| v3 — LLM Summaries | Close the vocabulary gap |
| v3 — Results & Findings | Summary across all versions + next steps |
| Conclusions | What's next |

## Eval Set

**Source:** `qa_pairs/solvers_qa.json` — 10 pairs covering `kernelpack.solvers`.  
**Generator:** Codex, prompted to write questions a scientist would ask if they
knew what they wanted numerically but didn't know the KernelPack API.  
**Tiers:** api (4), workflow (3), conceptual (3).  
**Validation:** Codex-generated. Ground truth unconfirmed — treat recall numbers
as directional only until Varun reviews.

**Fixed across all versions:**
- tree-sitter AST chunking at `function_definition` boundaries
- UniXcoder dense embeddings (ChromaDB, L2)
- BM25 sparse leg, whitespace-tokenized
- RRF fusion (k=60), top-3 returned
- Failure mode diagnosis via wide retrieval (top-50, dense-only) after each version

## Setup

In [55]:
%pip install -q \
    sentence-transformers==5.5.0 \
    tree-sitter==0.25.2 \
    tree-sitter-python==0.25.0 \
    chromadb==1.5.9 \
    rank-bm25==0.2.2

Note: you may need to restart the kernel to use updated packages.


In [56]:
import json
import numpy as np
from pathlib import Path

import chromadb
import tree_sitter_python as tspython
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tree_sitter import Language, Parser

# ── paths ──────────────────────────────────────────────────────────────────────
# Assumes kernelpack-python is cloned in the same parent directory as this repo.
# If not, update this path to point at your local kernelpack-python/src/kernelpack.
REPO_PATH = Path("../../kernelpack-python/src/kernelpack")
if not REPO_PATH.exists():
    raise FileNotFoundError(
        f"kernelpack-python not found at {REPO_PATH.resolve()}. "
        "Clone it as a sibling directory: git clone https://github.com/ShankarLab/kernelpack-python"
    )

# ── retrieval config ───────────────────────────────────────────────────────────
TOP_K = 3   # results returned per query

In [57]:
# Load all .py files from the repo
def load_docs(repo_path: Path) -> list[dict]:
    docs = []
    for path in sorted(repo_path.rglob("*.py")):  # will include __init__.py files
        docs.append({"path": str(path), "text": path.read_text(encoding="utf-8")})
    print(f"Loaded {len(docs)} files from {repo_path}")
    return docs

docs = load_docs(REPO_PATH)

Loaded 34 files from ../../kernelpack-python/src/kernelpack


In [58]:
# CHUNKING
MIN_LINES = 5   # functions with fewer than 5 lines dropped (inclusive)

# tree-sitter parser setup
PY_LANGUAGE = Language(tspython.language())
parser_ts   = Parser(PY_LANGUAGE)

def get_class_name(node):
    parent = node.parent
    if parent and parent.type == "decorated_definition":
        parent = parent.parent
    if (parent and parent.type == "block"
            and parent.parent and parent.parent.type == "class_definition"):
        return parent.parent.child_by_field_name("name").text.decode("utf-8")
    return None

def extract_chunks(source: str, tree, path: str, min_lines: int = MIN_LINES):
    """Walk the AST; return (kept, dropped) split by min_lines threshold."""
    kept, dropped = [], []
    def walk(node):
        if node.type == "function_definition":
            start = node.start_point[0]
            end   = node.end_point[0]
            entry = {
                "path":          path,
                "function_name": node.child_by_field_name("name").text.decode("utf-8"),
                "class_name":    get_class_name(node),
                "text":          source[node.start_byte:node.end_byte],
                "start_line":    start,
                "end_line":      end,
            }
            (kept if (end - start + 1) >= min_lines else dropped).append(entry)
            return  # don't recurse into functions
        for child in node.children:
            walk(child)
    walk(tree.root_node)
    return kept, dropped

all_chunks, all_dropped = [], []
for doc in docs:
    tree = parser_ts.parse(doc["text"].encode("utf-8"))
    kept, dropped = extract_chunks(doc["text"], tree, doc["path"])
    all_chunks.extend(kept)
    all_dropped.extend(dropped)

print(f"Chunks kept   : {len(all_chunks)}")
print(f"Chunks dropped: {len(all_dropped)}  (functions < {MIN_LINES} lines)")

Chunks kept   : 337
Chunks dropped: 277  (functions < 5 lines)


## Embedding Model & ChromaDB Client

Initialized once here and shared across all index versions.

In [59]:
model = SentenceTransformer("microsoft/unixcoder-base")
model.max_seq_length = 512
ef = SentenceTransformerEmbeddingFunction(model_name="microsoft/unixcoder-base")
ef._model = model

client = chromadb.EphemeralClient()
print("Embedding model and ChromaDB client ready.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 72047.17it/s]


Embedding model and ChromaDB client ready.


## Helper Functions

Run this cell once. All experiment cells below depend on these functions.

| Function | Purpose |
|----------|---------|
| `make_chunk_id(chunk)` | Stable chunk ID: relative path + line range |
| `make_metadata(chunk)` | Standard metadata dict for ChromaDB storage |
| `build_index(chunks, name, documents)` | Build ChromaDB collection + BM25 index |
| `retrieve(query, index, k, silent)` | Hybrid BM25 + dense retrieval with RRF |
| `is_hit(result_meta, target_file, target_sym)` | Exact-match hit check |
| `run_eval(qa_pairs, index, k)` | Run recall@k; return (hits, totals) by tier |
| `print_recall(hits, totals, label)` | Print recall breakdown by tier |
| `eval_query(qa, index, k)` | Print per-query retrieval results + verdict |
| `compare_versions(qa_pairs, idx_a, idx_b, ...)` | Print hit/miss changes between versions |
| `diagnose_wide(qa_pairs, index, n)` | Dense-only top-n diagnostic (isolates embedding quality) |

In [60]:
# ── ID and metadata ───────────────────────────────────────────────────────────

def make_chunk_id(chunk: dict) -> str:
    """Stable ID: relative file path + line range."""
    fp = str(Path(chunk["path"]).relative_to(REPO_PATH))
    return f"{fp}:{chunk['start_line']}-{chunk['end_line']}"


def make_metadata(chunk: dict) -> dict:
    """Standard metadata dict stored alongside each chunk in ChromaDB."""
    return {
        "file_path":     str(Path(chunk["path"]).relative_to(REPO_PATH)),
        "function_name": chunk["function_name"],
        "class_name":    chunk["class_name"] or "",
        "start_line":    chunk["start_line"],
        "end_line":      chunk["end_line"],
    }


# ── Index builder ──────────────────────────────────────────────────────────────

def build_index(
    chunks: list[dict],
    collection_name: str,
    documents: list[str] | None = None,
) -> dict:
    """Build a ChromaDB collection + BM25 index from a list of chunks.

    Args:
        chunks:          Output of extract_chunks.
        collection_name: Name for the ChromaDB collection.
        documents:       Pre-built document strings (e.g. summary-enriched text).
                         If None, raw chunk source text is used.
    Returns:
        dict with keys: col (ChromaDB collection), bm25 (BM25Okapi), ids (list[str]).
    """
    if documents is None:
        documents = [c["text"] for c in chunks]

    raw_ids   = [make_chunk_id(c) for c in chunks]
    metadatas = [make_metadata(c) for c in chunks]

    # deduplicate by ID
    seen = set()
    deduped_docs, deduped_metas, deduped_ids = [], [], []
    for doc, meta, cid in zip(documents, metadatas, raw_ids):
        if cid not in seen:
            seen.add(cid)
            deduped_docs.append(doc)
            deduped_metas.append(meta)
            deduped_ids.append(cid)

    col = client.get_or_create_collection(collection_name, embedding_function=ef)
    col.add(documents=deduped_docs, metadatas=deduped_metas, ids=deduped_ids)
    print(f"Collection {collection_name!r}: {col.count()} chunks")

    bm25 = BM25Okapi([doc.split() for doc in deduped_docs])
    return {"col": col, "bm25": bm25, "ids": deduped_ids}


# ── Retrieval ──────────────────────────────────────────────────────────────────

def retrieve(query: str, index: dict, k: int = TOP_K, silent: bool = False) -> list[dict]:
    """Hybrid BM25 + dense retrieval with RRF fusion.

    Args:
        query:  Query string.
        index:  Dict with keys col, bm25, ids — output of build_index.
        k:      Number of results to return.
        silent: If True, suppresses printed output.
    Returns:
        Ordered list of metadata dicts for the top-k chunks.
    """
    col  = index["col"]
    bm25 = index["bm25"]
    ids  = index["ids"]

    # dense leg — embed with UniXcoder, return top 10 by L2 distance
    dense_res = col.query(query_texts=[query], n_results=10)
    dense_ids = dense_res["ids"][0]
    dense_l2  = {cid: d for cid, d in zip(dense_res["ids"][0], dense_res["distances"][0])}

    # BM25 leg — scored by keyword overlap, return top 10
    bm25_scores = bm25.get_scores(query.split())
    top_bm25    = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:10]
    bm25_ids    = [ids[i] for i in top_bm25]

    # RRF fusion — score += 1/(rank + 60) for each list the chunk appears in
    # k=60 is the smoothing constant; dampens the advantage of being ranked 1st
    rrf: dict[str, float] = {}
    for rank, cid in enumerate(dense_ids):
        rrf[cid] = rrf.get(cid, 0.0) + 1 / (rank + 60)
    for rank, cid in enumerate(bm25_ids):
        rrf[cid] = rrf.get(cid, 0.0) + 1 / (rank + 60)

    top_ids = sorted(rrf, key=rrf.__getitem__, reverse=True)[:k]

    # col.get() returns results in input-ID order — reorder to match RRF ranking
    fetched   = col.get(ids=top_ids, include=["documents", "metadatas"])
    id_to_idx = {cid: i for i, cid in enumerate(fetched["ids"])}
    ordered_ids   = [cid for cid in top_ids if cid in id_to_idx]
    ordered_docs  = [fetched["documents"][id_to_idx[cid]] for cid in ordered_ids]
    ordered_metas = [fetched["metadatas"][id_to_idx[cid]] for cid in ordered_ids]

    if not silent:
        print(f'\nQUERY: "{query}"')
        print("=" * 60)
        for i, (cid, doc, meta) in enumerate(zip(ordered_ids, ordered_docs, ordered_metas), 1):
            cls    = f"{meta['class_name']}." if meta["class_name"] else ""
            l2_str = f"{dense_l2[cid]:.4f}" if cid in dense_l2 else "n/a (BM25-only)"
            print(f"\n  Rank {i} | RRF {rrf[cid]:.4f} | L2 {l2_str}")
            print(f"  {meta['file_path']} — {cls}{meta['function_name']} (lines {meta['start_line']}–{meta['end_line']})")
            print("  " + "-" * 56)
            print("  " + doc[:300].replace("\n", "\n  "))

    return ordered_metas


# ── Eval helpers ───────────────────────────────────────────────────────────────

def is_hit(result_meta: dict, target_file: str, target_sym: str) -> bool:
    """True if result_meta exactly matches the expected file and function name."""
    return (
        result_meta["file_path"].split("/")[-1] == target_file
        and result_meta["function_name"] == target_sym
    )


def run_eval(qa_pairs: list[dict], index: dict, k: int = TOP_K) -> tuple[dict, dict]:
    """Run recall@k over qa_pairs against index. Returns (hits, totals) by tier."""
    hits   = {"api": 0, "workflow": 0, "conceptual": 0}
    totals = {"api": 0, "workflow": 0, "conceptual": 0}
    for qa in qa_pairs:
        results     = retrieve(qa["query"], index, k=k, silent=True)
        target_file = qa["source_file"].split("/")[-1]
        target_sym  = qa["source_symbol"].split(".")[-1]
        totals[qa["tier"]] += 1
        if any(is_hit(r, target_file, target_sym) for r in results):
            hits[qa["tier"]] += 1
    return hits, totals


def print_recall(hits: dict, totals: dict, label: str = "") -> None:
    """Print recall breakdown by tier and overall."""
    header = f"── Recall@{sum(totals.values())} by tier"
    if label:
        header += f" — {label}"
    print(header)
    for tier in ["api", "workflow", "conceptual"]:
        h, t = hits[tier], totals[tier]
        pct  = round(100 * h / t) if t else 0
        print(f"  {tier:<12} {h}/{t}  ({pct}%)")
    total_h = sum(hits.values())
    total_t = sum(totals.values())
    print(f"  {'overall':<12} {total_h}/{total_t}  ({round(100 * total_h / total_t) if total_t else 0}%)")


def eval_query(qa: dict, index: dict, k: int = TOP_K) -> None:
    """Print per-query retrieval results and hit/miss verdict."""
    results     = retrieve(qa["query"], index, k=k, silent=True)
    target_file = qa["source_file"].split("/")[-1]
    target_sym  = qa["source_symbol"].split(".")[-1]
    hit         = any(is_hit(r, target_file, target_sym) for r in results)

    print(f"\n{'='*70}")
    print(f"QUERY : {qa['query']}")
    print(f"TIER  : {qa['tier']}")
    print(f"TARGET: {qa['source_symbol']} ({qa['source_file']})")
    print(f"ANSWER: {qa['expected_answer']}")
    print(f"{'─'*70}")
    for i, r in enumerate(results, 1):
        print(f"  Rank {i}: {r['file_path']} — {r['function_name']}")
    print(f"{'─'*70}")
    print(f"VERDICT: {'✓ HIT' if hit else '✗ MISS'}")


def compare_versions(
    qa_pairs: list[dict],
    index_a: dict,
    index_b: dict,
    label_a: str = "a",
    label_b: str = "b",
    k: int = TOP_K,
) -> None:
    """Print queries where hit/miss status changed between two index versions."""
    for qa in qa_pairs:
        target_file = qa["source_file"].split("/")[-1]
        target_sym  = qa["source_symbol"].split(".")[-1]
        hit_a = any(is_hit(r, target_file, target_sym)
                    for r in retrieve(qa["query"], index_a, k=k, silent=True))
        hit_b = any(is_hit(r, target_file, target_sym)
                    for r in retrieve(qa["query"], index_b, k=k, silent=True))
        if hit_a != hit_b:
            status = "✓ -> ✗ REGRESSED" if hit_a else "✗ -> ✓ RECOVERED"
            print(f"{status}: {qa['source_symbol']}")


# ── Wide retrieval diagnostic ──────────────────────────────────────────────────

def diagnose_wide(qa_pairs: list[dict], index: dict, n: int = 50) -> None:
    """Dense-only wide retrieval diagnostic.

    For each query, reports:
      target_rank  — where the target chunk appears in dense-only top-n
      referenced   — whether the target symbol appears in any of the top-10 chunks

    Uses dense-only retrieval (no BM25 / RRF) to isolate embedding quality.
    A target in the top-10 neighborhood but outside top-3 in the full hybrid
    run is a ranking problem, not a vocabulary gap.
    """
    col = index["col"]
    for qa in qa_pairs:
        target_sym  = qa["source_symbol"].split(".")[-1]
        target_file = qa["source_file"].split("/")[-1]

        results_wide = col.query(query_texts=[qa["query"]], n_results=n)
        all_metas    = results_wide["metadatas"][0]

        target_rank = next(
            (i + 1 for i, m in enumerate(all_metas)
             if m["file_path"].split("/")[-1] == target_file
             and m["function_name"] == target_sym),
            None,
        )

        top10_docs = results_wide["documents"][0][:10]
        referenced = any(target_sym in doc for doc in top10_docs)

        print(f"{qa['source_symbol']}")
        print(f"  target rank        : {target_rank or 'not found'}")
        print(f"  referenced in top-10: {referenced}")


print("Helper functions defined.")

Helper functions defined.


## Load Eval Data

QA pairs are loaded once here and reused across all versions.

In [61]:
with open("qa_pairs/solvers_qa.json") as f:
    qa_pairs_solvers = json.load(f)

with open("qa_pairs/benchmark_qa.json") as f:
    qa_pairs_benchmark = json.load(f)

print(f"Solvers QA pairs  : {len(qa_pairs_solvers)}")
print(f"Benchmark QA pairs: {len(qa_pairs_benchmark)}")

Solvers QA pairs  : 10
Benchmark QA pairs: 3


## v1 — Baseline

**What's fixed at this version:**
- MIN_LINES=5 — functions shorter than 5 lines dropped
- 273 chunks indexed, raw source text only

**Known gaps (motivate v2 and v3):**
- `set_nonlinear_tolerance` and `set_linear_tolerance` are 2-line functions —
  never indexed, guaranteed misses
- No plain-English descriptions — vocabulary gap between NL queries and
  code identifiers

In [62]:
idx_v1 = build_index(all_chunks, "kp-hybrid-v1")

Collection 'kp-hybrid-v1': 337 chunks


In [63]:
for qa in qa_pairs_solvers:
    eval_query(qa, idx_v1)


QUERY : After running a pure Neumann elliptic benchmark, how can I record whether the constant-mode constraint was actually applied?
TIER  : api
TARGET: PoissonSolver.last_solve_used_nullspace (src/kernelpack/solvers/poisson.py)
ANSWER: The stationary Poisson path treats the problem as pure Neumann when the maximum absolute Dirichlet coefficient is at most 1e-13. In that case it augments the sparse system with an all-ones row and column, stores the boolean on the solver, and returns the same information as `used_nullspace_augmentation` plus a `lagrange_multiplier` in the result dictionary. The stored boolean can be queried afterward through the nullspace-status accessor.
──────────────────────────────────────────────────────────────────────
  Rank 1: solvers/pu_sl_fd_advection_diffusion.py — enable_mass_constraint
  Rank 2: solvers/diffusion.py — set_initial_state
  Rank 3: solvers/pu_sl_pu_advection_diffusion.py — enable_mass_constraint
───────────────────────────────────────────────

In [64]:
hits_v1, totals_v1 = run_eval(qa_pairs_solvers, idx_v1)
print_recall(hits_v1, totals_v1, "v1 — solvers")

── Recall@10 by tier — v1 — solvers
  api          0/4  (0%)
  workflow     0/3  (0%)
  conceptual   1/3  (33%)
  overall      1/10  (10%)


In [65]:
for qa in qa_pairs_benchmark:
    eval_query(qa, idx_v1)

hits_bench_v1, totals_bench_v1 = run_eval(qa_pairs_benchmark, idx_v1)
print_recall(hits_bench_v1, totals_bench_v1, "v1 — benchmark")


QUERY : how to create an RBF-FD differentiation matrix?
TIER  : workflow
TARGET: FDDiffOp.assemble_op (src/kernelpack/rbffd/core.py)
ANSWER: Use the RBF-FD classes in `kernelpack.rbffd`. Create stencil settings with `StencilProperties.from_accuracy(operator=..., convergence_order=..., dimension=..., approximation="rbf", tree_mode="all", point_set="interior_boundary")`, create operator settings with `OpProperties(...)`, then instantiate `FDDiffOp(lambda: RBFStencil())`. Call `assemble_op(domain, op_name, st_props, op_props, neu_coeff=None, dir_coeff=None, active_rows=None)`, where `op_name` is a string such as `"lap"`, `"grad"`, or `"bc"` as supported by the stencil code. Retrieve the sparse CSR differentiation matrix with `get_op()`. For overlapped assembly, the source also provides `FDODiffOp` with the same `assemble_op` and `get_op` workflow.
──────────────────────────────────────────────────────────────────────
  Rank 1: divfree/core.py — DivFreeGram
  Rank 2: divfree/core.py — DFP

In [66]:
# see what got dropped during chunking
all_functions = [m["function_name"] for m in idx_v1["col"].get(include=["metadatas"])["metadatas"]]

targets = [
    "set_nonlinear_tolerance",
    "set_periodic_boundary",
    "set_linear_tolerance",
    "bdf1_step",
]

for target in targets:
    print(target, "→", "IN INDEX" if target in all_functions else "MISSING")

set_nonlinear_tolerance → MISSING
set_periodic_boundary → IN INDEX
set_linear_tolerance → MISSING
bdf1_step → IN INDEX


In [67]:
# Dense-only wide retrieval diagnostic (top-50)
# Separates "found the right neighborhood" from "found the exact target"
diagnose_wide(qa_pairs_solvers, idx_v1)

PoissonSolver.last_solve_used_nullspace
  target rank        : not found
  referenced in top-10: False
VariablePoissonSolver.solve
  target rank        : 40
  referenced in top-10: True
NonlinearVariablePoissonSolver.get_last_residual_norm
  target rank        : not found
  referenced in top-10: False
PUSLAdvectionSolver.get_solve_stats
  target rank        : not found
  referenced in top-10: False
DiffusionSolver.set_state_history
  target rank        : 1
  referenced in top-10: True
MultiSpeciesDiffusionSolver.set_initial_state
  target rank        : 4
  referenced in top-10: True
PUSLFDAdvectionDiffusionSolver.bdf3_step
  target rank        : 2
  referenced in top-10: True
build_stencil_properties
  target rank        : 2
  referenced in top-10: True
pu_patch_weight
  target rank        : 29
  referenced in top-10: True
IncompressibleEulerBDFBackend._get_cached_system
  target rank        : 15
  referenced in top-10: True


## v1 — Result

recall@3: **1/10 (10%)**

| Tier | Hits |
|------|------|
| api | 0/4 |
| workflow | 0/3 |
| conceptual | 1/3 |

Wide diagnostic (top-50): 4 exact hits, 3 reranker problems, 3 not indexed.
This is the floor every subsequent version is measured against.

## v1 — Failure Mode Analysis

Ran wide retrieval (top-50) on all 10 eval pairs to classify each miss.

| Target | Rank@50 | Referenced in top-10 | Category |
|--------|---------|----------------------|----------|
| PoissonSolver.last_solve_used_nullspace | not found | ✗ | Not indexed |
| VariablePoissonSolver.solve | 40 | ✓ | **Reranker** problem |
| NonlinearVariablePoissonSolver.get_last_residual_norm | not found | ✗ | Not indexed |
| PUSLAdvectionSolver.get_solve_stats | not found | ✗ | Not indexed |
| DiffusionSolver.set_state_history | 1 | ✓ | Hit |
| MultiSpeciesDiffusionSolver.set_initial_state | 4 | ✓ | Hit |
| PUSLFDAdvectionDiffusionSolver.bdf3_step | 2 | ✓ | Hit |
| build_stencil_properties | 2 | ✓ | Hit |
| pu_patch_weight | 29 | ✓ | **Reranker** problem |
| IncompressibleEulerBDFBackend._get_cached_system | 15 | ✓ | **Reranker** problem |

### Key finding
- 4 hits — target chunk surfaces at or near rank 1–4
- 3 reranker problems — chunk exists and is referenced nearby, but ranks 15–40
- 3 not indexed — chunk absent from the index entirely and not cross-referenced

The reranker finding holds from v1: retrieval is working. The bottleneck is ranking,
not recall. A cross-encoder reranker re-scores candidates by reading query and chunk
together — it would plausibly recover the 3 ranked-but-buried targets.

### Not-indexed failures
Three targets are absent from the index and unreferenced in top-10.
These are not a reranker problem. Likely causes: function too small to produce
a standalone chunk, or not captured at AST boundaries. Reranking cannot fix this.

## Tuning Plan

### Confirmed failure modes (from v1 wide diagnostic)

1. **Not indexed** — 3 targets absent from the index entirely.
   - `PoissonSolver.last_solve_used_nullspace`
   - `NonlinearVariablePoissonSolver.get_last_residual_norm`
   - `PUSLAdvectionSolver.get_solve_stats`

2. **Reranker problem** — chunk exists and is referenced in top-10 but ranks outside top-3.
   - `VariablePoissonSolver.solve` (rank 40)
   - `pu_patch_weight` (rank 29)
   - `IncompressibleEulerBDFBackend._get_cached_system` (rank 15)

### Experiment sequence

| Version | Change | Failure mode targeted |
|---------|--------|-----------------------|
| v2 | MIN_LINES=1 | not-indexed misses |
| v3a | solvers-only scope (control) | isolate index noise effect |
| v3 | LLM summaries | vocabulary gap |
| v4 (next) | cross-encoder reranker | reranker-problem misses — highest leverage |

## v2 — MIN_LINES=1
**Change:** Drop minimum line filter from 5 to 1
**Hypothesis:** Recovers set_nonlinear_tolerance and set_linear_tolerance

In [68]:
all_chunks_v2, all_dropped_v2 = [], []
for doc in docs:
    tree = parser_ts.parse(doc["text"].encode("utf-8"))
    kept, dropped = extract_chunks(doc["text"], tree, doc["path"], min_lines=1)
    all_chunks_v2.extend(kept)
    all_dropped_v2.extend(dropped)

print(f"v1 chunks: {len(all_chunks)}")
print(f"v2 chunks: {len(all_chunks_v2)}")

idx_v2 = build_index(all_chunks_v2, "kp-hybrid-v2")

# verify targets now exist
all_fns_v2 = [m["function_name"] for m in idx_v2["col"].get(include=["metadatas"])["metadatas"]]
for target in ["set_nonlinear_tolerance", "set_linear_tolerance"]:
    print(target, "→", "IN INDEX" if target in all_fns_v2 else "STILL MISSING")

v1 chunks: 337
v2 chunks: 614
Collection 'kp-hybrid-v2': 614 chunks
set_nonlinear_tolerance → IN INDEX
set_linear_tolerance → IN INDEX


In [69]:
hits_v2, totals_v2 = run_eval(qa_pairs_solvers, idx_v2)

print("── v1 vs v2 recall@3 ─────────────────")
print(f"  v1 overall: {sum(hits_v1.values())}/10 ({round(100 * sum(hits_v1.values()) / 10)}%)")
print(f"  v2 overall: {sum(hits_v2.values())}/10 ({round(100 * sum(hits_v2.values()) / 10)}%)")

── v1 vs v2 recall@3 ─────────────────
  v1 overall: 1/10 (10%)
  v2 overall: 2/10 (20%)


In [70]:
compare_versions(qa_pairs_solvers, idx_v1, idx_v2, "v1", "v2")

✗ -> ✓ RECOVERED: NonlinearVariablePoissonSolver.get_last_residual_norm


In [71]:
diagnose_wide(qa_pairs_solvers, idx_v2)

PoissonSolver.last_solve_used_nullspace
  target rank        : not found
  referenced in top-10: False
VariablePoissonSolver.solve
  target rank        : not found
  referenced in top-10: False
NonlinearVariablePoissonSolver.get_last_residual_norm
  target rank        : 2
  referenced in top-10: True
PUSLAdvectionSolver.get_solve_stats
  target rank        : not found
  referenced in top-10: False
DiffusionSolver.set_state_history
  target rank        : 7
  referenced in top-10: True
MultiSpeciesDiffusionSolver.set_initial_state
  target rank        : 7
  referenced in top-10: True
PUSLFDAdvectionDiffusionSolver.bdf3_step
  target rank        : 7
  referenced in top-10: True
build_stencil_properties
  target rank        : 2
  referenced in top-10: True
pu_patch_weight
  target rank        : 42
  referenced in top-10: True
IncompressibleEulerBDFBackend._get_cached_system
  target rank        : 19
  referenced in top-10: True


In [72]:
# what does the added noise look like?
new_chunks = [c for c in all_chunks_v2 if (c["end_line"] - c["start_line"] + 1) < 5]
for c in new_chunks[:20]:
    lines = c["end_line"] - c["start_line"]
    print(f"{lines} lines | {c['class_name']}.{c['function_name']}")
    print("  " + c["text"][:120].replace("\n", "\n  "))
    print()

2 lines | None._stack_field
  def _stack_field(U: np.ndarray) -> np.ndarray:
      U = np.asarray(U, dtype=float)
      return np.concatenate([U[:, d] for

3 lines | None._unstack_field
  def _unstack_field(rhs: np.ndarray, dim: int) -> np.ndarray:
      rhs = np.asarray(rhs, dtype=float).reshape(-1)
      n = 

3 lines | LocalDivFreeInterpolator._query_stencil_indices
  def _query_stencil_indices(self, point: np.ndarray) -> np.ndarray:
          idx = self.Tree.query(point, k=self.StencilSi

3 lines | LocalDivFreeInterpolator.assign_centers
  def assign_centers(self, Xq: np.ndarray) -> np.ndarray:
          Xq = np.asarray(Xq, dtype=float)
          idx = self.Tree

1 lines | DomainDescriptor.set_outer_level_set
  def set_outer_level_set(self, level_set: object) -> None:
          self.outer_level_set = level_set

1 lines | DomainDescriptor.set_sep_rad
  def set_sep_rad(self, sep_rad: float) -> None:
          self.sep_rad = float(sep_rad)

1 lines | DomainDescriptor.set_boundary_leve

In [73]:
# v2 benchmark eval
hits_bench_v2, totals_bench_v2 = run_eval(qa_pairs_benchmark, idx_v2)

n = len(qa_pairs_benchmark)
print("── Benchmark recall@3 ────────────────")
print(f"  v1: {sum(hits_bench_v1.values())}/{n}")
print(f"  v2: {sum(hits_bench_v2.values())}/{n}")
print("  v3: n/a — solvers-only index, benchmark comparison not valid")

── Benchmark recall@3 ────────────────
  v1: 0/3
  v2: 0/3
  v3: n/a — solvers-only index, benchmark comparison not valid


## v2 — Result

recall@3: **2/10 (20%)** — up from 1/10. `NonlinearVariablePoissonSolver.get_last_residual_norm` recovered to rank 2.

The improvement came with a tradeoff: adding short functions introduced noise that degraded ranks for 5 previously-stable targets. Line count is a proxy for "substantial enough to embed well" — not a reliable filter for target inclusion.

## v2 — Failure Mode Analysis

Wide retrieval (top-50, dense leg only). Compared against v1 baseline.

| Target | v1 rank | v2 rank | top-10 | Category |
|--------|---------|---------|--------|----------|
| PoissonSolver.last_solve_used_nullspace | not found | not found | ✗ | Not indexed |
| VariablePoissonSolver.solve | 40 | not found | ✗ | **Regression** |
| NonlinearVariablePoissonSolver.get_last_residual_norm | not found | 2 | ✓ | Recovered |
| PUSLAdvectionSolver.get_solve_stats | not found | not found | ✗ | Not indexed |
| DiffusionSolver.set_state_history | 1 | 7 | ✓ | Degraded |
| MultiSpeciesDiffusionSolver.set_initial_state | 4 | 7 | ✓ | Degraded |
| PUSLFDAdvectionDiffusionSolver.bdf3_step | 2 | 7 | ✓ | Degraded |
| build_stencil_properties | 2 | 2 | ✓ | Hit |
| pu_patch_weight | 29 | 42 | ✓ | Degraded |
| IncompressibleEulerBDFBackend._get_cached_system | 15 | 19 | ✓ | Degraded |

- recall@3 improved (1→2) because NonlinearVariablePoisson recovered to rank 2.
- Most previously-ranked targets degraded slightly in rank but stayed in top-10
  — they still pass soft recall, just with less margin.
- VariablePoissonSolver.solve is the only hard regression: rank 40 + referenced
  in v1, completely absent in v2. Check whether the v2 change affected
  chunking or indexing for that target specifically.
- Not-indexed failures unchanged. Not a retrieval or ranking problem.

## v3a — Solvers-only (Control)

**Change:** Scope index to `kernelpack.solvers` only — no summaries.  
**Hypothesis:** Smaller, focused index reduces noise independently of summary enrichment.  
**Purpose:** Isolates scoping effect so v3's summary contribution can be measured cleanly.

In [74]:
# solvers-only chunk list — shared by v3a and v3
all_chunks_v3 = [c for c in all_chunks if "solvers" in c["path"]]
print(f"Solvers chunks: {len(all_chunks_v3)}")

Solvers chunks: 187


In [75]:
# v3a — solvers-only index, no summaries
# control experiment: isolates whether scoping to solvers alone improves recall
# independent of summary enrichment
idx_v3a = build_index(all_chunks_v3, "kp-hybrid-v3a")

hits_v3a, totals_v3a = run_eval(qa_pairs_solvers, idx_v3a)

print("── v1 vs v3a recall@3 ────────────────")
print(f"  v1  (full index, no summaries)   : {sum(hits_v1.values())}/10")
print(f"  v3a (solvers only, no summaries) : {sum(hits_v3a.values())}/10")

Collection 'kp-hybrid-v3a': 187 chunks
── v1 vs v3a recall@3 ────────────────
  v1  (full index, no summaries)   : 1/10
  v3a (solvers only, no summaries) : 2/10


## v3a — Result

recall@3: **2/10 (20%)** — same as v2. Scoping to solvers alone did not improve recall.

Confirms the v3 gain is attributable to summaries, not the smaller index.

## v3 — LLM Summaries

**Change:** Prepend Codex-generated plain-English summary to each solver chunk.  
**Hypothesis:** Bridging the vocabulary gap between NL queries and code identifiers recovers buried targets.

In [76]:
with open("metadata/solvers_summaries.json") as f:
    summaries = json.load(f)

# key = (function_name, class_name)
summary_lookup = {
    (s["function_name"], s["class_name"] or ""): s["summary"]
    for s in summaries
}
print(f"Loaded {len(summary_lookup)} summaries")

Loaded 366 summaries


In [77]:
# build enriched documents for v3 — prepend LLM summary to each chunk
documents_v3 = []
for c in all_chunks_v3:
    key     = (c["function_name"], c["class_name"] or "")
    summary = summary_lookup.get(key, "")
    enriched = f"{summary}\n\n{c['text']}" if summary else c["text"]
    documents_v3.append(enriched)

matched_v3 = sum(1 for c in all_chunks_v3
                 if (c["function_name"], c["class_name"] or "") in summary_lookup)
print(f"{matched_v3}/{len(all_chunks_v3)} chunks matched a summary")

187/187 chunks matched a summary


In [78]:
idx_v3 = build_index(all_chunks_v3, "kp-hybrid-v3", documents=documents_v3)

# verify target functions are present
all_functions_v3 = [m["function_name"] for m in idx_v3["col"].get(include=["metadatas"])["metadatas"]]
targets = ["set_nonlinear_tolerance", "set_periodic_boundary", "set_linear_tolerance", "bdf1_step"]
for target in targets:
    print(target, "→", "IN INDEX" if target in all_functions_v3 else "MISSING")

# sanity check: confirm summary is prepended to raw source
sample = idx_v3["col"].get(limit=1, include=["documents"])
print("\n── Sample enriched document (first 500 chars) ──")
print(sample["documents"][0][:500])

Collection 'kp-hybrid-v3': 187 chunks
set_nonlinear_tolerance → MISSING
set_periodic_boundary → IN INDEX
set_linear_tolerance → MISSING
bdf1_step → IN INDEX

── Sample enriched document (first 500 chars) ──
Translates a solver accuracy/order request and domain dimension into the stencil size, polynomial degree, spline degree, tree mode, and point set used by the RBF-FD assembler.

def build_stencil_properties(domain: DomainDescriptor, xi: int, theta: int, point_set: str) -> StencilProperties:
    # The solver layer asks for target order `xi` and operator order `theta`;
    # here we translate that request into a concrete stencil size and
    # polynomial degree that the rbffd layer understands.
   


In [79]:
hits_v3, totals_v3 = run_eval(qa_pairs_solvers, idx_v3)

print("── v1 vs v3a vs v3 recall@3 ──────────")
print(f"  v1  (full index, no summaries)     : {sum(hits_v1.values())}/10")
print(f"  v3a (solvers only, no summaries)   : {sum(hits_v3a.values())}/10")
print(f"  v3  (solvers only, with summaries) : {sum(hits_v3.values())}/10")

── v1 vs v3a vs v3 recall@3 ──────────
  v1  (full index, no summaries)     : 1/10
  v3a (solvers only, no summaries)   : 2/10
  v3  (solvers only, with summaries) : 3/10


In [80]:
compare_versions(qa_pairs_solvers, idx_v1, idx_v3, "v1", "v3")

✗ -> ✓ RECOVERED: DiffusionSolver.set_state_history
✗ -> ✓ RECOVERED: pu_patch_weight


In [81]:
diagnose_wide(qa_pairs_solvers, idx_v3)

PoissonSolver.last_solve_used_nullspace
  target rank        : not found
  referenced in top-10: True
VariablePoissonSolver.solve
  target rank        : 18
  referenced in top-10: True
NonlinearVariablePoissonSolver.get_last_residual_norm
  target rank        : not found
  referenced in top-10: False
PUSLAdvectionSolver.get_solve_stats
  target rank        : not found
  referenced in top-10: False
DiffusionSolver.set_state_history
  target rank        : 4
  referenced in top-10: True
MultiSpeciesDiffusionSolver.set_initial_state
  target rank        : 2
  referenced in top-10: True
PUSLFDAdvectionDiffusionSolver.bdf3_step
  target rank        : 6
  referenced in top-10: True
build_stencil_properties
  target rank        : 1
  referenced in top-10: True
pu_patch_weight
  target rank        : 1
  referenced in top-10: True
IncompressibleEulerBDFBackend._get_cached_system
  target rank        : 7
  referenced in top-10: True


## v3 — Results & Findings

### Recall@3
| Version | Change | recall@3 |
|---------|--------|----------|
| v1  | full index, no summaries (baseline) | 1/10 (10%) |
| v2  | MIN_LINES=1 | 2/10 (20%) |
| v3a | solvers only, no summaries | 2/10 (20%) |
| v3  | solvers only, with summaries | 3/10 (30%) |

Both scoping and summaries contributed independently. v3a beat v1 by 1 —
scoping to solvers reduced index noise. v3 beat v3a by 1 — summaries
closed the vocabulary gap on top of that.

### The real finding
8/10 targets now appear in the top-10 retrieved neighborhood. In v1 that
number was effectively 4/10 for exact hits. The vocabulary gap is mostly
closed. The bottleneck is now ranking.

### Recoveries vs regressions
- **Recovered:** VariablePoissonSolver.solve (not found in v2 → rank 18 in v3).
  Still a reranker problem, but at least the chunk is findable again.
- **New regression:** NonlinearVariablePoissonSolver.get_last_residual_norm
  was rank 2 in v2, now not found and unreferenced. This was v2's strongest
  result — losing it in v3 is the most significant failure here. Needs
  investigation before moving forward.
- **Big gains:** pu_patch_weight (42 → 1), build_stencil_properties (2 → 1),
  MultiSpeciesDiffusionSolver.set_initial_state (7 → 2). Summaries gave
  the embeddings something to match against.

## v3 — Failure Mode Analysis

Wide retrieval (top-50, dense leg only). Compared against v1 and v2 baselines.

| Target | v1 rank | v2 rank | v3 rank | v3 top-10 | Category |
|--------|---------|---------|---------|-----------|----------|
| PoissonSolver.last_solve_used_nullspace | not found | not found | not found | ✓ | Cross-ref only |
| VariablePoissonSolver.solve | 40 | not found | 18 | ✓ | Reranker problem |
| NonlinearVariablePoissonSolver.get_last_residual_norm | not found | 2 | not found | ✗ | **Regression** |
| PUSLAdvectionSolver.get_solve_stats | not found | not found | not found | ✗ | Not indexed |
| DiffusionSolver.set_state_history | 1 | 7 | 4 | ✓ | Hit |
| MultiSpeciesDiffusionSolver.set_initial_state | 4 | 7 | 2 | ✓ | Hit |
| PUSLFDAdvectionDiffusionSolver.bdf3_step | 2 | 7 | 6 | ✓ | Hit |
| build_stencil_properties | 2 | 2 | 1 | ✓ | Hit |
| pu_patch_weight | 29 | 42 | 1 | ✓ | Hit |
| IncompressibleEulerBDFBackend._get_cached_system | 15 | 19 | 7 | ✓ | Hit |

- Six targets now rank in the top-10 as exact chunks — up from four in v1.
- PoissonSolver.last_solve_used_nullspace moved from completely dark to
  cross-referenced. The chunk is still absent but neighboring context now
  mentions it. Chunking gap, not retrieval gap.
- NonlinearVariablePoissonSolver.get_last_residual_norm is the critical
  regression: rank 2 in v2, gone in v3 with no cross-reference. Either the
  summary for that chunk degraded its embedding, or the chunk was displaced
  by a summary chunk during indexing. Check the index directly.
- pu_patch_weight went from rank 42 in v2 to rank 1 in v3 — the clearest
  demonstration that summary enrichment works for math-heavy targets.

# Conclusions
### What's still broken
- **Not indexed:** PUSLAdvectionSolver.get_solve_stats — absent and
  unreferenced across all three versions. Not a retrieval problem.
- **Reranker problems:** VariablePoissonSolver.solve (rank 18),
  PoissonSolver.last_solve_used_nullspace (not found but referenced) —
  neighborhood retrieval is working, ranking is not.
- **Unexplained regression:** NonlinearVariablePoissonSolver.get_last_residual_norm
  needs a root cause before v4.

### Next
Cross-encoder reranker. 8/10 targets are in the top-10 window. A reranker
re-scores each candidate by reading the query and chunk together — it would
plausibly recover the buried targets. Investigate the NonlinearVariablePoisson
regression in parallel.